# 3. The Agent: 5 Questions It Can Answer Accurately

The agent (`src/solarout/agent.py`) is a deterministic tool-calling dispatcher (no external LLM key required -- see README) over the tool functions in `src/solarout/tools.py`. **Missing weather handling**: when a question doesn't specify expected weather, the agent falls back to that city's long-run historical average conditions for the relevant month (climatology), explicitly disclosing which inputs were assumed rather than guessing silently or refusing to answer.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from solarout.agent import interpret
from solarout.tools import SolarTools

tools = SolarTools()

## Q1: Expected output tomorrow given conditions

**Question:** _What is my expected solar output tomorrow in Darwin if it's cloudy and rainy?_

**Why this is accurate:** Uses the **full** model (a real cloud/rain signal was supplied) which has holdout R^2 ~ 0.97 -- the dominant driver (Sunshine/Cloud) is exactly what was provided, so this is the most accurate question type the agent can answer.

In [2]:
print(interpret("What is my expected solar output tomorrow in Darwin if it's cloudy and rainy?", tools))

Expected solar output for Darwin in June: 2.13 kWh/kWp/day (range 1.93-2.32), using the 'full' model (holdout R^2=0.97). I assumed MinTemp, MaxTemp, Humidity9am, Humidity3pm, Pressure9am, Pressure3pm, WindSpeed9am, WindSpeed3pm from Darwin's historical average for June, since you didn't specify them.


## Q2: Farm sizing for a given capacity

**Question:** _If I set up a solar farm in Alice Springs of size 2 MW, what's my expected daily output?_

**Why this is accurate:** Output per kWp is the same well-validated **full**-model quantity from Q1 (climatology-backed when weather isn't specified), simply scaled linearly by installed capacity -- linear scaling introduces no additional error beyond the per-kWp prediction itself.

In [3]:
print(interpret("If I set up a solar farm in Alice Springs of size 2 MW, what's my expected daily output?", tools))

A 2000 kW farm in Alice Springs in October is expected to produce about 9366 kWh/day (range 8976-9756 kWh), using the 'full' model (holdout R^2=0.97). (assumed MinTemp, MaxTemp, Rainfall, Humidity9am, Humidity3pm, Pressure9am, Pressure3pm, WindSpeed9am, WindSpeed3pm, Sunshine, Cloud9am, Cloud3pm from Alice Springs's historical average for October, since you didn't specify them)


## Q3: Best month for output in a city

**Question:** _Which month is Hobart likely to have the highest solar output?_

**Why this is accurate:** Answered directly from the Global Solar Atlas long-term climatology baseline (no regression model needed) -- by definition the most reliable possible answer, since it's the actual multi-year satellite-derived average for that city/month.

In [4]:
print(interpret('Which month is Hobart likely to have the highest solar output?', tools))

For Hobart, the best month is historically January, with an average baseline output of 4.68 kWh/kWp/day (Global Solar Atlas long-term climatology). Full ranking: January=4.68, December=4.47, February=4.37, October=4.23, November=4.13, March=3.87, September=3.63, April=3.18, August=3.04, July=2.60, May=2.53, June=2.37


## Q4: Rainfall/cloud sensitivity vs a clear day

**Question:** _How much will cloud cover reduce my expected output compared to a clear day in Perth?_

**Why this is accurate:** Computed analytically from the same clear-sky-factor formula used to build the training target (not via model extrapolation, since cloud-only rows are a minority of training data) -- exact by construction, and the model's own holdout R^2 (~0.97) on rows where Sunshine *is* present confirms this formula's predictive validity.

In [5]:
print(interpret('How much will cloud cover reduce my expected output compared to a clear day in Perth?', tools))

For Perth in June, expected output falls from 3.51 kWh/kWp/day on a clear day (0 oktas) to 1.05 kWh/kWp/day fully overcast (8 oktas) -- a 70% reduction. At moderate cloud (4 oktas) the reduction is 35%. (Derived directly from the clear-sky-factor formula used to build the training target.)


## Q5: Output under typical/assumed conditions, with confidence range

**Question:** _Given typical conditions in Brisbane in October, what's my expected output?_

**Why this is accurate:** No weather given, so the agent falls back to the city's all-years historical climatology (disclosed explicitly) and reports a 1-sigma confidence range from the **full** model's holdout residual std -- giving an honest, quantified uncertainty band rather than a bare number.

In [6]:
print(interpret("Given typical conditions in Brisbane in October, what's my expected output?", tools))

Expected solar output for Brisbane in October: 3.25 kWh/kWp/day (range 3.06-3.45), using the 'full' model (holdout R^2=0.97). I assumed MinTemp, MaxTemp, Rainfall, Humidity9am, Humidity3pm, Pressure9am, Pressure3pm, WindSpeed9am, WindSpeed3pm, Sunshine, Cloud9am, Cloud3pm from Brisbane's historical average for October, since you didn't specify them.


## Extensibility

Additional questions (e.g. comparing two cities, multi-day farm output totals) are straightforward to add since they reuse the same `SolarTools` functions -- `estimate_farm_output` already supports a `days` parameter, and a city-comparison question would just call `predict_daily_output` twice. During the live presentation, the panel is welcome to try further questions via `interpret(<question>, tools)`.